<a href="https://colab.research.google.com/github/Lawrenceku/Revres/blob/main/predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import numpy as np

from sklearn.tree import export_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [50]:
df = pd.read_csv("irrigation_prediction.csv")

In [51]:
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Rainfed,Reservoir,4.73,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Canal,Groundwater,12.22,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Drip,Reservoir,5.52,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Canal,Reservoir,1.43,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,Canal,River,2.52,No,60.86,South,Medium


In [52]:
df = df[["Soil_pH", "Soil_Moisture", "Temperature_C", "Irrigation_Need"]] #drop unneeded columns

In [53]:
df.isna().sum(axis=0)  #check for null values

,0
Soil_pH,0
Soil_Moisture,0
Temperature_C,0
Irrigation_Need,0


In [54]:
df.describe()

,Soil_pH,Soil_Moisture,Temperature_C
count,10000.000000,10000.000000,10000.000000
mean,6.487857,36.969207,26.991423
std,0.979963,16.430845,8.664074
min,4.800000,8.000000,12.000000
25%,5.640000,22.860000,19.460000
50%,6.470000,37.240000,27.090000
75%,7.350000,50.940000,34.500000
max,8.200000,65.000000,42.000000


In [55]:
df["Irrigation_Need"].count()

np.int64(10000)

In [56]:
df.head()

,Soil_pH,Soil_Moisture,Temperature_C,Irrigation_Need
0,6.14,36.48,21.90,Low
1,6.41,50.56,36.50,Medium
2,7.71,40.07,41.83,Low
3,5.96,12.75,37.22,Medium
4,7.76,18.58,22.38,Medium


In [57]:
X, y = df.drop("Irrigation_Need", axis=1), df["Irrigation_Need"]

In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [59]:
encoder = OrdinalEncoder()

In [60]:
y_train = y_train.to_frame()
y_test = y_test.to_frame()

In [61]:
encoder.fit_transform(y_train)

array([[1.],
       [2.],
       [2.],
       ...,
       [2.],
       [2.],
       [1.]])

In [62]:
encoder.transform(y_test)

array([[2.],
       [1.],
       [0.],
       ...,
       [2.],
       [1.],
       [1.]])

In [63]:
y_test =pd.DataFrame(encoder.transform(y_test), columns = ['Irrigation_Need'])

In [64]:
y_train =pd.DataFrame(encoder.transform(y_train), columns = ['Irrigation_Need'])

In [65]:
tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)

In [66]:
tree_clf.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=3, random_state=42)

In [67]:
tree_rules = export_text(tree_clf,
                         feature_names=["Temp","Moisture","pH"])

print(tree_rules)

|--- Moisture <= 24.98
|   |--- pH <= 29.91
|   |   |--- Moisture <= 11.35
|   |   |   |--- class: 2.0
|   |   |--- Moisture >  11.35
|   |   |   |--- class: 2.0
|   |--- pH >  29.91
|   |   |--- pH <= 41.62
|   |   |   |--- class: 2.0
|   |   |--- pH >  41.62
|   |   |   |--- class: 0.0
|--- Moisture >  24.98
|   |--- pH <= 30.16
|   |   |--- Temp <= 4.96
|   |   |   |--- class: 1.0
|   |   |--- Temp >  4.96
|   |   |   |--- class: 1.0
|   |--- pH >  30.16
|   |   |--- Moisture <= 58.57
|   |   |   |--- class: 1.0
|   |   |--- Moisture >  58.57
|   |   |   |--- class: 1.0



In [68]:
y_pred = tree_clf.predict(X_test)

In [69]:
accuracy_score(y_pred, y_test)

0.6915